# Phase 34 — v1.2 Executable Harness & Task-Conditioning Binding
## Reviewer Notebook

**Classification:** `READ_ONLY_PHASE34_V1_2_EXECUTABLE_HARNESS_AND_TASK_BINDING_REVIEWER_NOTEBOOK`
**Revision:** `ra_medfilm_barr_phase34_efficiency_profiling_v1_2`

v1.0 = specification only · v1.1 = semantics repair only · first v1.1 execution attempt =
correctly **BLOCKED_PREMODEL**. No model was loaded, constructed, deserialized or forwarded,
and no runtime/memory measurement exists. v1.2 replaces only the non-executable harness and
the missing synthetic task-conditioning binding.


## 0. Setup

In [1]:
import hashlib
import json
from pathlib import Path

import pandas as pd

ROOT = Path(r"D:\Softfors\jup")
AUD = ROOT / "reports" / "audits"
TAB = ROOT / "reports" / "tables"


def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(1 << 20), b""):
            h.update(c)
    return h.hexdigest()


def jload(p):
    return json.loads(Path(p).read_text(encoding="utf-8"))


def tshow(name, n=None):
    df = pd.read_csv(TAB / name)
    print(name, "| shape:", df.shape)
    if n is not None:
        df = df.head(n)
    with pd.option_context("display.max_columns", None, "display.width", 240):
        print(df.to_string(index=False))
    return df


print("Phase34 v1.2 executable-harness + synthetic task-conditioning binding review")
print("READ-ONLY. No checkpoint load, no model construction, no forward, no timing,")
print("no memory measurement, no profiler, no protected data, no TEST, no training.")


Phase34 v1.2 executable-harness + synthetic task-conditioning binding review
READ-ONLY. No checkpoint load, no model construction, no forward, no timing,
no memory measurement, no profiler, no protected data, no TEST, no training.


## 1. v1.1 blocked attempt — preserved

In [2]:
bt = jload(AUD / "phase34_execution_blocked_terminal_v1_1.json")
print("terminal:", bt["terminal"])
print("status:", bt["status"])
print("blockers:", bt["blockers"])
print("measurements_performed:", bt["measurements_performed"])
pf11 = jload(AUD / "phase34_execution_source_preflight_v1_1.json")
print("v1.1 preflight pass:", pf11["preflight_pass"])
print("v1.1 missing executable behaviours:",
      pf11["static_semantic_audit"]["required_executable_behaviours_missing"])


terminal: PHASE34_EXECUTION_BLOCKED_PREMODEL__V1_1_FROZEN_SOURCE_OR_RUNTIME_SEMANTICS_MISMATCH__NO_MEASUREMENTS_PERFORMED
status: BLOCKED_PREMODEL
blockers: ['V1_1_WORKER_DOES_NOT_IMPLEMENT_REQUIRED_EXECUTABLE_RUNTIME_SEMANTICS', 'REQUIRED_TASK_DOMAIN_CONDITIONING_NOT_DETERMINISTICALLY_BOUND']
measurements_performed: 0
v1.1 preflight pass: False
v1.1 missing executable behaviours: ['G_model_eval', 'G_autocast_enabled', 'G_autocast_dtype_float16', 'G_no_grad', 'constructs_models', 'loads_checkpoints', 'runs_forwards', 'synchronized_cuda_timing', 'memory_measurement']


## 2. v1.2 blocker record

In [3]:
br = jload(AUD / "phase34_v1_1_execution_harness_blocker_record.json")
print("BLOCKER_1:", br["BLOCKER_1"])
print("BLOCKER_2:", br["BLOCKER_2"])
print("v1.1 source hashes were correct:", br["v1_1_source_hashes_were_correct"])
print("v1.1 measurements exist:", br["v1_1_measurements_exist"])
print("repair before first measurement:", br["repair_occurs_before_first_measurement"])


BLOCKER_1: V1_1_WORKER_DOES_NOT_IMPLEMENT_REQUIRED_EXECUTABLE_RUNTIME_SEMANTICS
BLOCKER_2: REQUIRED_TASK_DOMAIN_CONDITIONING_NOT_DETERMINISTICALLY_BOUND
v1.1 source hashes were correct: True
v1.1 measurements exist: False
repair before first measurement: True


## 3. v1.2 static source-semantic audit (AST only)

In [4]:
pf = jload(AUD / "phase34_execution_source_preflight_v1_2.json")
print("inspection:", pf["method_of_inspection"])
print("checks:", pf["n_checks"], "| failed:", pf["n_failed"],
      "| pass:", pf["static_preflight_pass"])
print()
for k, v in pf["checks"].items():
    print(f"  {'PASS' if v else 'FAIL'}  {k}")


inspection: AST_PARSE_AND_SHA256_HASHING_ONLY
checks: 33 | failed: 0 | pass: True

  PASS  actual_model_constructor_code_present
  PASS  actual_strict_checkpoint_load_code_present
  PASS  strict_true_present
  PASS  strict_missing_and_unexpected_keys_checked
  PASS  model_eval_calls_present
  PASS  parameter_count_verification_present
  PASS  G_autocast_enabled
  PASS  G_autocast_dtype_float16
  PASS  G_no_grad
  PASS  G_not_wrapped_in_inference_mode_in_primary_path
  PASS  R_psi_inference_mode_no_autocast
  PASS  B_phi_inference_mode_no_autocast
  PASS  S_omega_inference_mode_no_autocast
  PASS  G_autocast_does_not_wrap_other_components
  PASS  task_ids_accepted_by_G_builder
  PASS  domain_task_registry_explicitly_encoded
  PASS  batch1_task_schedule_present
  PASS  batch8_balanced_task_ids_present
  PASS  S_omega_valid_onehot_construction_present
  PASS  B_phi_structured_fields_bound
  PASS  cuda_events_timing_present
  PASS  cuda_synchronization_present
  PASS  throughput_denominato

## 4. Domain → task-id mapping

In [5]:
tc = jload(AUD / "phase34_synthetic_task_conditioning_contract_v1_2.json")
print("authority line:", tc["registry_authority_line"])
for d, v in tc["domain_task_registry"].items():
    print(f"  {d:8s} -> task_id {v['task_id']}  ({v['dtype']}, shape {v['shape_rule']}, "
          f"device {v['device_rule']})")
print()
print("authority:", tc["domain_task_registry"]["ISIC"]["source_authority"])
print("inferred/arbitrary task id permitted during measurement:",
      tc["inferred_or_arbitrary_task_id_permitted_during_measurement"])


authority line: TASK_ID = {"ISIC": 0, "BUSI": 1, "KVASIR": 2, "KUB": 3}
  ISIC     -> task_id 0  (torch.long / int64, shape [B], device same CUDA device as the component input)
  BUSI     -> task_id 1  (torch.long / int64, shape [B], device same CUDA device as the component input)
  KVASIR   -> task_id 2  (torch.long / int64, shape [B], device same CUDA device as the component input)
  KUB      -> task_id 3  (torch.long / int64, shape [B], device same CUDA device as the component input)

authority: src/analysis/multidomain_harmonization.py
inferred/arbitrary task id permitted during measurement: False


## 5. Batch-1 task schedule and batch-8 task vector

In [6]:
sc = jload(AUD / "phase34_synthetic_task_schedule_contract_v1_2.json")
print("batch1 schedule:", sc["batch1_task_schedule"])
print("batch1 order:", sc["batch1_task_order"])
print("per 200-iteration block:", sc["per_200_iteration_block"])
print("conditions per task id across 5 blocks:", sc["across_5_blocks_conditions_per_task_id"])
print("total measured iterations:", sc["total_measured_iterations"])
print("warmup iterations:", sc["warmup_iterations"], "| same cycle:",
      sc["warmup_uses_same_deterministic_cycle"],
      "| weighting interpretation:", sc["warmup_has_scientific_weighting_interpretation"])
print()
print("batch8 task vector:", sc["batch8_task_vector"], sc["batch8_task_vector_dtype"])
print("batch8 exactly balanced:", sc["batch8_is_exactly_balanced_four_domain"])
print("batch8 reproduces protected Phase30 membership:",
      sc["batch8_reproduces_protected_phase30_batch_membership"])
print()
print("batch1 classification:", sc["batch1_classification"])
print("batch8 classification:", sc["batch8_classification"])


batch1 schedule: task_id(iteration_index) = iteration_index mod 4
batch1 order: {'0': 'ISIC', '1': 'BUSI', '2': 'KVASIR', '3': 'KUB'}
per 200-iteration block: {'ISIC': 50, 'BUSI': 50, 'KVASIR': 50, 'KUB': 50}
conditions per task id across 5 blocks: 250
total measured iterations: 1000
warmup iterations: 50 | same cycle: True | weighting interpretation: False

batch8 task vector: [0, 1, 2, 3, 0, 1, 2, 3] int64
batch8 exactly balanced: True
batch8 reproduces protected Phase30 membership: False

batch1 classification: CANONICAL_GRAPH_HARDWARE_MICROBENCHMARK__BATCH1_NOT_CANONICAL_G_NUMERICAL_REPRODUCTION
batch8 classification: SYNTHETIC_HARDWARE_THROUGHPUT_BENCHMARK


## 6. Structured synthetic feature conditioning

In [7]:
sf = jload(AUD / "phase34_synthetic_structured_feature_contract_v1_2.json")
print("S_omega:")
print("  numeric indices:", sf["S_omega"]["numeric_indices"])
print("  one-hot indices:", sf["S_omega"]["onehot_indices"])
print("  all-20-as-randn:", sf["S_omega"]["all_20_as_unconstrained_randn"])
print("B_phi:")
print("  numeric dim:", sf["B_phi"]["numeric_feature_dim"])
print("  vector contains categorical/one-hot:", sf["B_phi"]["feature_vector_contains_categorical_or_onehot_fields"])
print("  B_PHI_EXPLICIT_TASK_CONDITIONING:", sf["B_phi"]["B_PHI_EXPLICIT_TASK_CONDITIONING"])
print("R_psi:")
print("  R_PSI_EXPLICIT_TASK_CONDITIONING:", sf["R_psi"]["R_PSI_EXPLICIT_TASK_CONDITIONING"])
print("G:")
print("  G_TASK_CONDITIONING:", sf["G"]["G_TASK_CONDITIONING"])
gen = jload(AUD / "phase34_synthetic_generator_terminology_v1_2.json")
print()
print("generator:", gen["generator_construction"], "->", gen["terminology"])
print("NumPy PCG64 used:", gen["numpy_pcg64_used"])


S_omega:
  numeric indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15]
  one-hot indices: [16, 17, 18, 19]
  all-20-as-randn: False
B_phi:
  numeric dim: 202
  vector contains categorical/one-hot: False
  B_PHI_EXPLICIT_TASK_CONDITIONING: INTEGER_TASK_ID_EMBEDDING
R_psi:
  R_PSI_EXPLICIT_TASK_CONDITIONING: NONE
G:
  G_TASK_CONDITIONING: INTEGER_TASK_ID_FILM_EMBEDDING

generator: torch.Generator(device="cuda").manual_seed(42) -> PYTORCH_CUDA_GENERATOR_SEED_42
NumPy PCG64 used: False


## 7. Timed-operation registry

In [8]:
tr = jload(AUD / "phase34_timed_operation_registry_v1_2.json")
for arm, v in tr["arms"].items():
    print(f"  {arm}")
    print(f"      scope    = {v['scope']}")
    print(f"      included = {v['included_operations']}")
    print(f"      preconstructed_inputs={v['input_tensors_preconstructed']} "
          f"checkpoint_load_included={v['checkpoint_load_included']} "
          f"fusion_included={v['fusion_included']} "
          f"decision_logic_included={v['decision_logic_included']}")
print()
print("END_TO_END_PIPELINE_RUNTIME used:", tr["end_to_end_pipeline_runtime_used"])


  GLOBAL_ONLY_C1
      scope    = MODEL_COMPUTE_PATH_RUNTIME
      included = ['G model forward', 'G sigmoid']
      preconstructed_inputs=True checkpoint_load_included=False fusion_included=False decision_logic_included=False
  FINAL_ROUTED_A8_C9
      scope    = MODEL_COMPUTE_PATH_RUNTIME
      included = ['G model forward', 'G sigmoid', 'B_phi model forward', 'R_psi model forward', 'R_psi sigmoid', 'S_omega model forward']
      preconstructed_inputs=True checkpoint_load_included=False fusion_included=False decision_logic_included=False
  SKIP_CONTROLLED_PATH
      scope    = MODEL_COMPUTE_PATH_RUNTIME
      included = ['G model forward', 'G sigmoid', 'B_phi model forward']
      preconstructed_inputs=True checkpoint_load_included=False fusion_included=False decision_logic_included=False
  EXECUTE_CONTROLLED_PATH
      scope    = MODEL_COMPUTE_PATH_RUNTIME
      included = ['G model forward', 'G sigmoid', 'B_phi model forward', 'R_psi model forward', 'R_psi sigmoid', 'S_omega model 

## 8. v1.2 source freeze

In [9]:
sfz = jload(AUD / "phase34_preexecution_source_freeze_v1_2.json")
for k, v in sfz["frozen_sources"].items():
    print(f"  {v['sha256'][:16]}...  {k:18s} {v['path']}")
print()
print("v1.1 source freeze preserved:", sfz["v1_1_source_freeze_preserved"])
print("v1.1 blocked attempt preserved:", sfz["v1_1_blocked_attempt_preserved"])
print("v1.2 supersedes v1.1 for future execution:",
      sfz["v1_2_supersedes_v1_1_for_future_execution"])
print("v1.2 measurements already performed:", sfz["v1_2_measurements_already_performed"])


  ca74fe88a8314b60...  common             scripts/analysis/phase34_v1_2_common.py
  302decd1c72f970b...  worker             scripts/analysis/phase34_efficiency_profiling_worker_v1_2.py
  73d4d25f91097e81...  harness            scripts/analysis/phase34_efficiency_profiling_execution_v1_2.py
  23f75f73624c582d...  preflight          scripts/analysis/phase34_execution_preflight_v1_2.py
  c103c0d323c3809a...  notebook_builder   scripts/analysis/build_phase34_spec_v1_2_reviewer_notebook.py
  a04ff3e61b4d900f...  model_G            src/models/film/global_medfilm.py
  4fe0abde3ec3a183...  model_R_psi        src/models/barr/regional_refiner.py
  fc7c1bdf263ae455...  model_B_phi        scripts/analysis/train_phase25d_router.py
  98cf958a377e2de3...  model_S_omega      scripts/analysis/phase25f_execution_common.py

v1.1 source freeze preserved: True
v1.1 blocked attempt preserved: True
v1.2 supersedes v1.1 for future execution: True
v1.2 measurements already performed: False


## 9. v1.2 execution eligibility

In [10]:
ea = jload(AUD / "phase34_execution_eligibility_adjudication_v1_2.json")
for k, v in ea["families"].items():
    print(f"  {k:52s} = {str(v['value']):5s}")
print()
print("first measured execution started:", ea["first_measured_execution_started"])


  G_component_runtime_eligible                         = True 
  R_psi_component_runtime_eligible                     = True 
  B_phi_component_runtime_eligible                     = True 
  S_omega_component_runtime_eligible                   = True 
  GLOBAL_ONLY_C1_runtime_eligible                      = True 
  FINAL_ROUTED_A8_C9_controlled_path_runtime_eligible  = True 
  SKIP_controlled_path_runtime_eligible                = True 
  batch1_latency_eligible                              = True 
  batch8_throughput_eligible                           = True 
  GPU_memory_eligible                                  = True 
  conditional_weighted_runtime_eligible                = True 
  synthetic_natural_route_replay_eligible              = False
  new_MAC_FLOP_profiler_eligible                       = False
  raw_ROUTER_HOLDOUT_execution_eligible                = False
  raw_GT_execution_eligible                            = False
  TEST_execution_eligible                              

## 10. Zero-execution audit

In [11]:
ne = jload(AUD / "phase34_v1_2_no_execution_audit.json")
for k, v in ne["counters"].items():
    print(f"  {k:50s} = {v}")
print("all zero:", ne["all_zero"])
assert ne["all_zero"]


  model_constructions                                = 0
  checkpoint_deserializations_into_model_objects     = 0
  model_loads                                        = 0
  model_forwards                                     = 0
  synthetic_forwards                                 = 0
  cuda_latency_iterations                            = 0
  throughput_measurements                            = 0
  gpu_inference_memory_measurements                  = 0
  flop_mac_profiler_executions                       = 0
  raw_image_reads                                    = 0
  raw_gt_reads                                       = 0
  router_holdout_raw_reads                           = 0
  test_reads                                         = 0
  training_steps                                     = 0
all zero: True


## 11. v1.2 final check and terminal

In [12]:
fc = jload(AUD / "phase34_specification_final_check_v1_2.json")
print("final check:", fc["n_checks"], "checks,", fc["n_failed"], "failed")
term = jload(AUD / "phase34_specification_terminal_v1_2.json")
print()
print("terminal:")
print(" ", term["terminal"])


final check: 37 checks, 0 failed

terminal:
  PHASE34_EFFICIENCY_PROFILING_SPEC_V1_2_COMPLETE__EXECUTABLE_RUNTIME_HARNESS_IMPLEMENTED__CANONICAL_COMPONENT_PRECISION_AND_STRICT_CHECKPOINT_LOADING_BOUND__FOUR_DOMAIN_SYNTHETIC_TASK_CONDITIONING_AND_STRUCTURED_FEATURE_INPUTS_FROZEN__SYNCHRONIZED_LATENCY_THROUGHPUT_MEMORY_AND_PROCESS_ISOLATION_PATHS_STATICALLY_VERIFIED__V1_2_SOURCE_FREEZE_COMPLETE__FIRST_MEASURED_EXECUTION_ELIGIBLE_NOT_STARTED


---
## Summary

| item | value |
|---|---|
| v1.1 execution attempt | correctly `BLOCKED_PREMODEL`, preserved |
| v1.2 static preflight | PASS (AST only, 0 failures) |
| domain→task_id | ISIC 0 · BUSI 1 · KVASIR 2 · KUB 3 |
| batch-1 schedule | `iteration_index mod 4`, 50/task/block, 250/task, 1000 total |
| batch-8 vector | `[0,1,2,3,0,1,2,3]` (balanced) |
| S_omega tail | valid one-hot at indices 16..19 |
| B_phi conditioning | integer task id → `Embedding(4,16)` |
| R_psi conditioning | NONE |
| generator | `PYTORCH_CUDA_GENERATOR_SEED_42` (not PCG64) |
| model constructions · loads · forwards · timing · memory | 0 · 0 · 0 · 0 · 0 |
| first measured execution | NOT started |
| TEST | `LOCKED_UNTIL_PHASE39` · Phase35 NOT started |
